# Deep NLP Project - QA Model V3

Version from-scratch avec améliorations: tokenizer robuste, tête no-answer, décodage span conjoint, early stopping, device Apple MPS.


## 1) Imports et Device


In [1]:
import os
import re
import json
import math
import random
import string
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Device:", device)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


Device: mps


## 2) Paramètres


In [2]:
data_path = "data/"
train_path = os.path.join(data_path, "train-v2.0.json")
dev_path = os.path.join(data_path, "dev-v2.0.json")

MAX_CONTEXT_LEN = 220
MAX_QUESTION_LEN = 40
MIN_FREQ = 2

BATCH_SIZE = 32
NUM_EPOCHS = 10
EMBED_DIM = 200
HIDDEN_SIZE = 128
LR = 2e-3
WEIGHT_DECAY = 1e-2
CLIP_NORM = 1.0
PATIENCE = 3

NO_ANSWER_WEIGHT = 0.7
MAX_ANSWER_LEN = 30
NO_ANSWER_THRESHOLD = 0.5

TRAIN_SUBSET = None   # ex: 30000
DEV_SUBSET = 3000     # None = full dev


## 3) Tokenization et Chargement SQuAD v2


In [3]:
TOKEN_PATTERN = re.compile(r"\w+|[^\w\s]")

def tokenize_advanced(text):
    return TOKEN_PATTERN.findall(text.lower())

def char_to_token_span(context, context_tokens, answer_start, answer_text):
    lowered = context.lower()
    offsets = []
    cursor = 0
    for tok in context_tokens:
        idx = lowered.find(tok, cursor)
        if idx == -1:
            idx = lowered.find(tok)
            if idx == -1:
                return None, None
        offsets.append((idx, idx + len(tok)))
        cursor = idx + len(tok)

    ans_start = answer_start
    ans_end = answer_start + len(answer_text)
    s_tok = e_tok = None
    for i, (s, e) in enumerate(offsets):
        if s <= ans_start < e:
            s_tok = i
        if s < ans_end <= e:
            e_tok = i
            break
    return s_tok, e_tok

def load_squad_v2(path, max_examples=None):
    with open(path, "r", encoding="utf-8") as f:
        squad = json.load(f)

    examples = []
    for article in squad["data"]:
        for paragraph in article["paragraphs"]:
            context = paragraph["context"]
            context_tokens = tokenize_advanced(context)
            for qa in paragraph["qas"]:
                question_tokens = tokenize_advanced(qa["question"])
                is_impossible = qa.get("is_impossible", False)

                if is_impossible:
                    examples.append({
                        "context_tokens": context_tokens,
                        "question_tokens": question_tokens,
                        "start": 0,
                        "end": 0,
                        "has_answer": 0,
                        "raw_answer": "",
                    })
                else:
                    answers = qa.get("answers", [])
                    if not answers:
                        continue
                    ans = answers[0]
                    s_tok, e_tok = char_to_token_span(context, context_tokens, ans["answer_start"], ans["text"].lower())
                    if s_tok is None or e_tok is None:
                        continue
                    examples.append({
                        "context_tokens": context_tokens,
                        "question_tokens": question_tokens,
                        "start": s_tok + 1,  # 0 réservé à <CLS>
                        "end": e_tok + 1,
                        "has_answer": 1,
                        "raw_answer": ans["text"],
                    })

                if max_examples is not None and len(examples) >= max_examples:
                    return examples
    return examples

train_examples = load_squad_v2(train_path, TRAIN_SUBSET)
dev_examples = load_squad_v2(dev_path, DEV_SUBSET)
print("Train:", len(train_examples), "| Dev:", len(dev_examples))


Train: 130309 | Dev: 3000


## 4) Vocabulaire et Encodage


In [4]:
def build_vocab(examples, min_freq=2):
    counter = Counter()
    for ex in examples:
        counter.update(ex["context_tokens"])
        counter.update(ex["question_tokens"])

    vocab = {"<PAD>": 0, "<UNK>": 1, "<CLS>": 2}
    for tok, freq in counter.items():
        if freq >= min_freq and tok not in vocab:
            vocab[tok] = len(vocab)
    return vocab

vocab = build_vocab(train_examples, MIN_FREQ)
VOCAB_SIZE = len(vocab)
print("Vocab size:", VOCAB_SIZE)

def encode(tokens, vocab_obj, max_len, add_cls=False):
    if add_cls:
        tokens = ["<CLS>"] + tokens
    ids = [vocab_obj.get(t, vocab_obj["<UNK>"]) for t in tokens][:max_len]
    return ids + [vocab_obj["<PAD>"]] * (max_len - len(ids))

def prepare_data(examples, vocab_obj):
    Xc, Xq, ys, ye, ya = [], [], [], [], []
    for ex in examples:
        if ex["start"] >= MAX_CONTEXT_LEN or ex["end"] >= MAX_CONTEXT_LEN:
            continue
        Xc.append(encode(ex["context_tokens"], vocab_obj, MAX_CONTEXT_LEN, add_cls=True))
        Xq.append(encode(ex["question_tokens"], vocab_obj, MAX_QUESTION_LEN, add_cls=False))
        ys.append(ex["start"])
        ye.append(ex["end"])
        ya.append(ex["has_answer"])
    return np.array(Xc), np.array(Xq), np.array(ys), np.array(ye), np.array(ya)

Xc_train, Xq_train, ys_train, ye_train, ya_train = prepare_data(train_examples, vocab)
Xc_dev, Xq_dev, ys_dev, ye_dev, ya_dev = prepare_data(dev_examples, vocab)
print("Train tensors:", Xc_train.shape, Xq_train.shape)
print("Dev tensors:", Xc_dev.shape, Xq_dev.shape)


Vocab size: 80404
Train tensors: (129302, 220) (129302, 40)
Dev tensors: (2993, 220) (2993, 40)


## 5) DataLoaders


In [5]:
train_ds = TensorDataset(
    torch.LongTensor(Xc_train),
    torch.LongTensor(Xq_train),
    torch.LongTensor(ys_train),
    torch.LongTensor(ye_train),
    torch.FloatTensor(ya_train),
)

dev_ds = TensorDataset(
    torch.LongTensor(Xc_dev),
    torch.LongTensor(Xq_dev),
    torch.LongTensor(ys_dev),
    torch.LongTensor(ye_dev),
    torch.FloatTensor(ya_dev),
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False)
print("Batches train/dev:", len(train_loader), len(dev_loader))


Batches train/dev: 4041 94


## 6) Modèle V3


In [ ]:
from models import QAModelV3

model_v3 = QAModelV3(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE).to(device)
print(model_v3.__class__.__name__)



## 7) Entraînement (Early Stopping)


In [7]:
optimizer = optim.AdamW(model_v3.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
ce_loss = nn.CrossEntropyLoss()
bce_loss = nn.BCEWithLogitsLoss()

def run_epoch(model, loader, optimizer_obj=None):
    train_mode = optimizer_obj is not None
    model.train() if train_mode else model.eval()
    total = 0.0
    for ctx, qst, y_s, y_e, y_has in loader:
        ctx, qst = ctx.to(device), qst.to(device)
        y_s, y_e, y_has = y_s.to(device), y_e.to(device), y_has.to(device)
        with torch.set_grad_enabled(train_mode):
            s_logits, e_logits, na_logit = model(ctx, qst)
            loss_span = ce_loss(s_logits, y_s) + ce_loss(e_logits, y_e)
            loss_na = bce_loss(na_logit, 1.0 - y_has)
            loss = loss_span + NO_ANSWER_WEIGHT * loss_na
            if train_mode:
                optimizer_obj.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_NORM)
                optimizer_obj.step()
        total += loss.item()
    return total / max(len(loader), 1)

best_val = float('inf')
pat_left = PATIENCE
for epoch in range(1, NUM_EPOCHS + 1):
    tr = run_epoch(model_v3, train_loader, optimizer)
    va = run_epoch(model_v3, dev_loader, None)
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | train={tr:.4f} | dev={va:.4f}")
    if va < best_val:
        best_val = va
        pat_left = PATIENCE
        torch.save(model_v3.state_dict(), 'bidaf_model_v3_best.pt')
        print("  -> best checkpoint saved")
    else:
        pat_left -= 1
        print("  -> no improvement, patience:", pat_left)
        if pat_left == 0:
            print("Early stopping")
            break


Epoch 01/10 | train=6.2211 | dev=5.1284
  -> best checkpoint saved
Epoch 02/10 | train=5.2570 | dev=5.0773
  -> best checkpoint saved
Epoch 03/10 | train=4.7590 | dev=5.5967
  -> no improvement, patience: 2
Epoch 04/10 | train=4.3920 | dev=5.6507
  -> no improvement, patience: 1
Epoch 05/10 | train=4.0921 | dev=6.2704
  -> no improvement, patience: 0
Early stopping


## 8) Évaluation EM/F1 + Décodage Conjoint + Post-traitement


In [8]:
def normalize_answer(s):
    s = s.lower()
    s = ''.join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    return ' '.join(s.split())

def compute_em(pred, truth):
    return int(normalize_answer(pred) == normalize_answer(truth))

def compute_f1(pred, truth):
    p = normalize_answer(pred).split()
    t = normalize_answer(truth).split()
    if not p and not t:
        return 1.0
    if not p or not t:
        return 0.0
    common = Counter(p) & Counter(t)
    n = sum(common.values())
    if n == 0:
        return 0.0
    prec, rec = n / len(p), n / len(t)
    return 2 * prec * rec / (prec + rec)

def clean_prediction(prediction):
    prediction = re.sub(r"^[\s\(\)\[\],;:\.""]+", "", prediction)
    prediction = re.sub(r"[\s\(\)\[\],;:\.""]+$", "", prediction)
    prediction = re.sub(r'\s+', ' ', prediction).strip()
    return prediction

@torch.no_grad()
def decode_best_span(start_logits, end_logits, max_answer_len=30):
    best_score = -1e30
    best_s, best_e = 0, 0
    L = start_logits.size(0)
    for s in range(L):
        e_max = min(L - 1, s + max_answer_len - 1)
        e_rel = torch.argmax(end_logits[s:e_max+1]).item()
        e = s + e_rel
        score = start_logits[s].item() + end_logits[e].item()
        if score > best_score:
            best_score = score
            best_s, best_e = s, e
    return best_s, best_e

@torch.no_grad()
def predict_answer_v3(model, context_tokens, question_tokens, vocab_obj, no_answer_threshold=0.5):
    model.eval()
    ctx = torch.LongTensor([encode(context_tokens, vocab_obj, MAX_CONTEXT_LEN, add_cls=True)]).to(device)
    qst = torch.LongTensor([encode(question_tokens, vocab_obj, MAX_QUESTION_LEN, add_cls=False)]).to(device)
    s_logits, e_logits, na_logit = model(ctx, qst)
    s_logits, e_logits = s_logits[0], e_logits[0]

    p_no = torch.sigmoid(na_logit[0]).item()
    if p_no >= no_answer_threshold:
        return '', 0, 0, p_no

    s_idx, e_idx = decode_best_span(s_logits, e_logits, MAX_ANSWER_LEN)
    if s_idx == 0 or e_idx == 0:
        return '', s_idx, e_idx, p_no

    s_txt = s_idx - 1
    e_txt = min(e_idx - 1, len(context_tokens)-1)
    if e_txt < s_txt:
        return '', s_idx, e_idx, p_no

    pred = ' '.join(context_tokens[s_txt:e_txt+1])
    return clean_prediction(pred), s_idx, e_idx, p_no

if os.path.exists('bidaf_model_v3_best.pt'):
    model_v3.load_state_dict(torch.load('bidaf_model_v3_best.pt', map_location=device))

em_total = 0
f1_total = 0.0
n = 0
for ex in dev_examples:
    if ex['start'] >= MAX_CONTEXT_LEN or ex['end'] >= MAX_CONTEXT_LEN:
        continue
    pred, _, _, _ = predict_answer_v3(model_v3, ex['context_tokens'], ex['question_tokens'], vocab, NO_ANSWER_THRESHOLD)
    truth = ex['raw_answer']
    em_total += compute_em(pred, truth)
    f1_total += compute_f1(pred, truth)
    n += 1

print('='*90)
print('MODEL V3 - FINAL EVAL')
print('N =', n)
print(f'EM = {100*em_total/max(n,1):.2f}%')
print(f'F1 = {100*f1_total/max(n,1):.2f}%')
print('='*90)


MODEL V3 - FINAL EVAL
N = 2993
EM = 37.19%
F1 = 39.21%


## 9) Exemples Qualitatifs


In [11]:
for i in range(min(5, len(dev_examples))):
    ex = dev_examples[i]
    pred, s, e, p_no = predict_answer_v3(model_v3, ex['context_tokens'], ex['question_tokens'], vocab, NO_ANSWER_THRESHOLD)
    print(f"Exemple {i+1}")
    print("Q:", ' '.join(ex['question_tokens']))
    print("True:", ex['raw_answer'] if ex['raw_answer'] else '[NO_ANSWER]')
    print("Pred:", pred if pred else '[NO_ANSWER]')
    print(f"p_no_answer={p_no:.3f}, span=({s},{e})")
    print('-'*90)


Exemple 1
Q: in what country is normandy located ?
True: France
Pred: france
p_no_answer=0.050, span=(35,35)
------------------------------------------------------------------------------------------
Exemple 2
Q: when were the normans in normandy ?
True: 10th and 11th centuries
Pred: 10th
p_no_answer=0.070, span=(22,22)
------------------------------------------------------------------------------------------
Exemple 3
Q: from which countries did the norse originate ?
True: Denmark, Iceland and Norway
Pred: france
p_no_answer=0.111, span=(35,35)
------------------------------------------------------------------------------------------
Exemple 4
Q: who was the norse leader ?
True: Rollo
Pred: charles iii of west francia
p_no_answer=0.226, span=(74,78)
------------------------------------------------------------------------------------------
Exemple 5
Q: what century did the normans first gain their separate identity ?
True: 10th century
Pred: 10th
p_no_answer=0.049, span=(128,128)
-----

## 10) Sauvegarde


In [12]:
torch.save(model_v3.state_dict(), 'bidaf_model_v3_last.pt')
print('Saved: bidaf_model_v3_last.pt')
if os.path.exists('bidaf_model_v3_best.pt'):
    print('Best: bidaf_model_v3_best.pt')


Saved: bidaf_model_v3_last.pt
Best: bidaf_model_v3_best.pt


## 11) Comparaison Finale des 3 Modèles (Tableau)

Cette cellule compare **Model v1**, **Model v2 amélioré** et **Model v3** avec un tableau EM/F1.


In [ ]:
COMPARISON_MAX_EXAMPLES = 2000

# --- Métriques partagées ---

def _normalize_answer_cmp(s):
    s = s.lower()
    s = ''.join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    return ' '.join(s.split())


def _em_cmp(pred, truth):
    return int(_normalize_answer_cmp(pred) == _normalize_answer_cmp(truth))


def _f1_cmp(pred, truth):
    p = _normalize_answer_cmp(pred).split()
    t = _normalize_answer_cmp(truth).split()
    if len(p) == 0 and len(t) == 0:
        return 1.0
    if len(p) == 0 or len(t) == 0:
        return 0.0
    common = Counter(p) & Counter(t)
    n = sum(common.values())
    if n == 0:
        return 0.0
    precision = n / len(p)
    recall = n / len(t)
    return 2 * precision * recall / (precision + recall)


# --- Pipeline Baseline (v1/v2) ---
from models import QAModelV1, QAModelV2

def tokenize_basic(text):
    return text.lower().split()


def load_squad_basic(path, max_examples=None):
    with open(path, 'r', encoding='utf-8') as f:
        squad = json.load(f)

    examples = []
    for article in squad['data']:
        for paragraph in article['paragraphs']:
            context = paragraph['context']
            context_tokens = tokenize_basic(context)
            for qa in paragraph['qas']:
                answer_obj = None
                if len(qa.get('answers', [])) > 0:
                    answer_obj = qa['answers'][0]
                elif len(qa.get('plausible_answers', [])) > 0:
                    answer_obj = qa['plausible_answers'][0]
                if answer_obj is None:
                    continue

                question_tokens = tokenize_basic(qa['question'])

                char_count = 0
                start_token = end_token = None
                for i, token in enumerate(context_tokens):
                    if char_count <= answer_obj['answer_start'] < char_count + len(token) + 1:
                        start_token = i
                    if char_count <= answer_obj['answer_start'] + len(answer_obj['text']) <= char_count + len(token) + 1:
                        end_token = i
                        break
                    char_count += len(token) + 1

                if start_token is not None and end_token is not None:
                    examples.append({
                        'context_tokens': context_tokens,
                        'question_tokens': question_tokens,
                        'start': start_token,
                        'end': end_token,
                        'raw_answer': answer_obj['text']
                    })

                if max_examples is not None and len(examples) >= max_examples:
                    return examples

    return examples


def build_vocab_basic(examples, min_freq=2):
    counter = Counter()
    for ex in examples:
        counter.update(ex['context_tokens'])
        counter.update(ex['question_tokens'])
    vocab_obj = {'<PAD>': 0, '<UNK>': 1}
    for token, freq in counter.items():
        if freq >= min_freq:
            vocab_obj[token] = len(vocab_obj)
    return vocab_obj


def encode_basic(tokens, vocab_obj, max_len):
    ids = [vocab_obj.get(t, vocab_obj['<UNK>']) for t in tokens][:max_len]
    return ids + [vocab_obj['<PAD>']] * (max_len - len(ids))


@torch.no_grad()
def predict_span_basic(model_obj, context_tokens, question_tokens, vocab_obj, max_context_len=200, max_question_len=30):
    model_obj.eval()
    ctx = torch.LongTensor([encode_basic(context_tokens, vocab_obj, max_context_len)]).to(device)
    qst = torch.LongTensor([encode_basic(question_tokens, vocab_obj, max_question_len)]).to(device)
    s_logits, e_logits = model_obj(ctx, qst)
    s_idx = torch.argmax(s_logits[0]).item()
    e_idx = torch.argmax(e_logits[0]).item()
    if e_idx < s_idx:
        e_idx = s_idx
    s_idx = min(s_idx, len(context_tokens) - 1)
    e_idx = min(e_idx, len(context_tokens) - 1)
    return ' '.join(context_tokens[s_idx:e_idx+1])


def eval_basic_model(model_obj, examples, vocab_obj, max_examples=2000):
    data = examples[:max_examples] if max_examples is not None else examples
    em_total = 0
    f1_total = 0.0
    n = 0
    for ex in data:
        pred = predict_span_basic(model_obj, ex['context_tokens'], ex['question_tokens'], vocab_obj)
        truth = ex['raw_answer']
        em_total += _em_cmp(pred, truth)
        f1_total += _f1_cmp(pred, truth)
        n += 1
    return {
        'n_eval': n,
        'em': 100.0 * em_total / max(n, 1),
        'f1': 100.0 * f1_total / max(n, 1),
    }


def eval_v3_model(max_examples=2000):
    data = dev_examples[:max_examples] if max_examples is not None else dev_examples
    em_total = 0
    f1_total = 0.0
    n = 0
    for ex in data:
        if ex['start'] >= MAX_CONTEXT_LEN or ex['end'] >= MAX_CONTEXT_LEN:
            continue
        pred, _, _, _ = predict_answer_v3(model_v3, ex['context_tokens'], ex['question_tokens'], vocab, NO_ANSWER_THRESHOLD)
        truth = ex['raw_answer']
        em_total += _em_cmp(pred, truth)
        f1_total += _f1_cmp(pred, truth)
        n += 1
    return {
        'n_eval': n,
        'em': 100.0 * em_total / max(n, 1),
        'f1': 100.0 * f1_total / max(n, 1),
    }


# --- Chargement des données/poids ---
results = []

if os.path.exists('bidaf_model.pt') and os.path.exists('bidaf_model_improved.pt'):
    print('Chargement et évaluation des modèles v1/v2...')
    train_basic = load_squad_basic(train_path)
    dev_basic = load_squad_basic(dev_path, max_examples=COMPARISON_MAX_EXAMPLES)
    vocab_basic = build_vocab_basic(train_basic, min_freq=2)

    v1 = QAModelV1(len(vocab_basic), 100, 128, 200).to(device)
    v1.load_state_dict(torch.load('bidaf_model.pt', map_location=device))
    m1 = eval_basic_model(v1, dev_basic, vocab_basic, COMPARISON_MAX_EXAMPLES)
    results.append({'model': 'Model v1', **m1})

    v2 = QAModelV2(len(vocab_basic), 100, 128, 200).to(device)
    v2.load_state_dict(torch.load('bidaf_model_improved.pt', map_location=device))
    m2 = eval_basic_model(v2, dev_basic, vocab_basic, COMPARISON_MAX_EXAMPLES)
    results.append({'model': 'Model v2 (amélioré)', **m2})
else:
    print('Checkpoints v1/v2 introuvables, comparaison partielle.')

if os.path.exists('bidaf_model_v3_best.pt'):
    print('Chargement et évaluation du modèle v3...')
    model_v3.load_state_dict(torch.load('bidaf_model_v3_best.pt', map_location=device))
    m3 = eval_v3_model(COMPARISON_MAX_EXAMPLES)
    results.append({'model': 'Model v3', **m3})
else:
    print('Checkpoint v3 introuvable: bidaf_model_v3_best.pt')


# --- Tableau ---
if len(results) == 0:
    print('Aucun résultat à afficher.')
else:
    print('Modèle | N évalués | EM (%) | F1 (%) |')
    print('|---|---:|---:|---:|')
    for r in results:
        print(f"| {r['model']} | {r['n_eval']} | {r['em']:.2f} | {r['f1']:.2f} |")

    best = sorted(results, key=lambda x: x['f1'], reverse=True)[0]
    print(f"Meilleur modèle (F1): {best['model']} ({best['f1']:.2f}%)")

Chargement et évaluation des modèles v1/v2...
